# Download and prep NISAR data

This shows how the NISAR data is downloaded and assembled into an Xarray data cube. 

In [ ]:
from shapely.geometry import box 
import py3dep 
import rasterio
from rasterio.enums import Resampling
import geopandas as gpd

import sys
import os
from pathlib import Path

# import from this repo 
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2
from scripts.download import download_nisar, build_cube, download_nisar_gslc
from scripts.layers import get_nlcd_layers 

First, we setup logging to get useful information and error messages. 

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True 
)

The track, frame, start and end dates are all known based on out study area. 

In [ ]:
track = 19
frame = 21
start = '2025-12-01'
end = '2026-04-30'
proc_code = 'P05023'

data_dir = "../data/"
area_file = data_dir + "aso/er_20260318/ASO_EastRiver_2026Mar18_swe_50m.tif"

with rasterio.open(area_file) as src:
    bounds = src.bounds         
    crs = src.crs
shp = gpd.GeoDataFrame(geometry=[box(*bounds)], crs=crs)

out = data_dir + "nisar/east_river"
layers = ['unwrapped_phase', 'coherence_unwrapped', # 80 m interferogram 
            'wrapped_phase', 'coherence_wrapped',  # 20 m interferogram 
            'incidence_angle', 'perpendicular_baseline',
            'reference_slant_range', 'secondary_slant_range'] # for spatial coherence  

gslc_layers = ['backscatter', 'NEBZ'] # for thermal coherence 

A function in scripts/download.py does the heavy lifting. If the files already exist, they'll be skipped. 

In [7]:
###################### downloads nisar data #######################
(grans, files) = download_nisar(
    track=track,
    frame=frame,
    start_date=start,
    end_date=end,
    aoi=shp,
    layers=layers,
    output_dir=out,
    crid=proc_code
)

###################### downloads nisar data #######################
(grans, files) = download_nisar_gslc(
    track=track,
    frame=frame,
    start_date=start,
    end_date=end,
    aoi=shp,
    layers=gslc_layers,
    output_dir=out,
    crid=proc_code
)

2026-10-06 21:12:26,099 - INFO - Searching ASF for NISAR GUNW granules: track=19, frame=21, 2025-12-01–2026-04-30
2026-10-06 21:12:26,116 - INFO - SEARCH: Using search opts {
    "frame": [
        21
    ],
    "processingLevel": [
        "GUNW"
    ],
    "relativeOrbit": [
        19
    ],
    "start": "2025-12-01T00:00:00Z",
    "end": "2026-04-30T00:00:00Z",
    "dataset": [
        "NISAR"
    ]
}
2026-10-06 21:12:26,122 - INFO - SEARCH: Using cmr endpoint: "https://cmr.earthdata.nasa.gov/search/granules.umm_json"
2026-10-06 21:12:26,123 - INFO - SUBQUERY 1: Beginning subquery with opts: {
    "frame": [
        21
    ],
    "processingLevel": [
        "GUNW"
    ],
    "relativeOrbit": [
        19
    ],
    "start": "2025-12-01T00:00:00Z",
    "end": "2026-04-30T00:00:00Z",
    "collections": [
        "C1256524081-ASFDEV",
        "C1256533420-ASFDEV",
        "C1256535568-ASFDEV",
        "C1256535570-ASFDEV",
        "C1256574227-ASFDEV",
        "C1256578011-ASFDEV",
 

2026-10-06 21:12:41,661 - INFO - BlockCache joined background fetch without waiting.
2026-10-06 21:12:41,663 - INFO - BlockCache fetching block (sync) 13
2026-10-06 21:12:42,430 - INFO - BlockCache fetching block (async) 14
2026-10-06 21:12:42,436 - INFO - BlockCache fetching block (sync) 12
2026-10-06 21:12:43,212 - INFO - BlockCache waiting for background fetch.
2026-10-06 21:12:43,459 - INFO - BlockCache fetching block (async) 16
2026-10-06 21:12:43,566 - INFO - BlockCache waiting for background fetch.
2026-10-06 21:12:44,424 - INFO - BlockCache fetching block (async) 17
2026-10-06 21:12:44,637 - INFO -   wrote NISAR_L2_PR_GUNW_017_019_A_021_018_4000_SH_20260404T121121_20260404T121156_20260416T121121_20260416T121156_P05023_N_F_J_001_secondary_slant_range.tif (67x85)
2026-10-06 21:12:44,638 - INFO -   NISAR_L2_PR_GUNW_016_019_A_021_017_4000_SH_20260323T121121_20260323T121156_20260404T121121_20260404T121156_P05023_N_F_J_001_unwrapped_phase.tif already exists, skipping
2026-10-06 21:12

Once it's downloaded, we can get a list of all the files that we have. 

In [ ]:
files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*')]

Build some data cubes from the tif files. 

In [ ]:

####################### builds coherence xarrays #######################
wrapped_files = [f for f in files if "coherence_wrapped" in str(f)]
unwrapped_files = [f for f in files if "coherence_unwrapped" in str(f)]
inc_files = [f for f in files if "incidence_angle" in str(f)]

nisar_20m = build_cube(wrapped_files, "coherence", utc_offset_hours=-7)     # 20 m grid
nisar_80m = build_cube(unwrapped_files, "coherence", utc_offset_hours=-7)  # 80 m grid
nisar_inc = build_cube(inc_files, "inc_angle", utc_offset_hours=-7)


Next, add a DEM and NLCD canopy cover layer. 

In [ ]:
####################### adds nlcd and dem data #######################
# --- Wrapped dataset (20 m grid) ---
crs_20m = nisar_20m.rio.crs
aoi_20m = box(*nisar_20m.rio.bounds())

nlcd_20m = get_nlcd_layers(tile_aoi=aoi_20m, crs=crs_20m, tile_ref_grid=nisar_20m)

dem_20m = py3dep.get_dem(geometry=aoi_20m, resolution=20, crs=crs_20m)
dem_20m = dem_20m.rio.write_crs(dem_20m.rio.crs).rio.reproject_match(nisar_20m, resampling=Resampling.bilinear)

nisar_20m = nisar_20m.merge(nlcd_20m)
nisar_20m['elevation'] = dem_20m

# --- Unwrapped dataset (80 m grid) ---
crs_80m = nisar_80m.rio.crs
aoi_80m = box(*nisar_80m.rio.bounds())

nlcd_80m = get_nlcd_layers(tile_aoi=aoi_80m, crs=crs_80m, tile_ref_grid=nisar_80m)

aoi_dem = aoi_80m.buffer(1000)
dem_80m = py3dep.get_dem(geometry=aoi_dem, resolution=80, crs=crs_80m)
dem_80m = dem_80m.rio.write_crs(dem_80m.rio.crs).rio.reproject_match(nisar_80m, resampling=Resampling.bilinear)

nisar_80m = nisar_80m.merge(nlcd_80m)
nisar_80m['elevation'] = dem_80m

Save the zarrs. 

In [ ]:
####################### saves zarrs #######################
nisar_20m.to_zarr(data_dir + 'nisar/zarrs/obs_20m.zarr', mode="w")
nisar_80m.to_zarr(data_dir + 'nisar/zarrs/obs_80m.zarr', mode="w")

Plot some of the data, so we know what we are working with. 

In [ ]:
nisar_80m['coherence'].plot(col='pair', col_wrap=5, cmap='inferno')
nisar_20m['coherence'].plot(col='pair', col_wrap=5, cmap='inferno')